## Melt analysis figures for Joint Western and Eastern Snow Conference 2026

Plots of melt timing based on iSnobal and terrain data  
Basins and water year coverage: Animas, Yampa, Jordan from WY 2022-2024  
Uses studio conda env  
Leverages eb_analysis.ipynb looking at snotel sites  
Includes processing single ds for all em files in water year in a basin

In [ ]:
import sys

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/env/')
import helpers as h

import seaborn as sns

In [ ]:
%reload_ext autoreload
%autoreload 2

## Set up files and directories

In [ ]:
workdir = '/uufs/chpc.utah.edu/common/home/skiles-group2/model_runs_jmh/thp'
script_dir = '/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/isnobal_scripts'

In [ ]:
basins = ['animas', 'yampa', 'jordan']
wy_list = [2022, 2023, 2024]

In [ ]:
# Use zarr to compress data
# Write once (after combining)
basin = 'animas'
wy = 2022
em_zarr_fn = f'/uufs/chpc.utah.edu/common/home/skiles-group2/model_runs_jmh/thp/zarr_stores/{basin}_unified_em_wy{wy}.zarr'
em_ds = xr.open_zarr(em_zarr_fn)
print(em_ds)

In [ ]:
# snow_zarr_fn = f'/uufs/chpc.utah.edu/common/home/skiles-group2/model_runs_jmh/thp/zarr_stores/{basin}_unified_snow_wy{wy}.zarr'
# snow_ds = xr.open_zarr(snow_zarr_fn)
# snow_ds

In [ ]:
# import geopandas as gpd
# # Plot the polygon on top
# poly_fn = '/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/ancillary/polys/jordan_32612.shp'
# poly_gdf = gpd.read_file(poly_fn)
# poly_gdf.plot()
# fig, ax = plt.subplots()
# snow_ds['thickness'][120, :, :].plot.imshow(ax=ax)
# poly_gdf.plot(ax=ax)
# # equal aspect
# ax.set_aspect('equal')

# # ok, using the polygon, extract a bounding box
# minx, miny, maxx, maxy = poly_gdf.total_bounds
# # Pad it a little bit to be safe, pixel sizes are ~100m, so round and pad by 200m (2 pixels)
# minx = np.floor(minx / 200) * 200 - 200
# maxx = np.ceil(maxx / 200) * 200 + 200
# miny = np.floor(miny / 200) * 200 - 200
# maxy = np.ceil(maxy / 200) * 200 + 200
# print(minx, miny, maxx, maxy)

# fig, ax = plt.subplots()
# snow_ds['thickness'][120, :, :].plot.imshow(ax=ax)
# poly_gdf.plot(ax=ax)
# # Plot that bounding box on top!
# rect = plt.Rectangle((minx, miny), maxx - minx, maxy - miny,
#                      edgecolor='red', facecolor='none')
# ax.add_patch(rect)
# # equal aspect
# ax.set_aspect('equal')

# # Save this bounding box for later use to crop datasets
# from shapely.geometry import box
# bbox_fn = f'/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/ancillary/polys/{basin}_smallbbox_32612.geojson'
# bbox_poly = box(minx, miny, maxx, maxy)
# bbox_gdf = gpd.GeoDataFrame({'geometry': [bbox_poly]},
#                             crs=poly_gdf.crs)
# bbox_gdf.to_file(bbox_fn, driver='GeoJSON')

In [ ]:
# Precompute basin mean variables for plotting
em_basin_mean = em_ds.mean(dim=['x', 'y']).compute()

In [ ]:
# Basin-mean snowmelt, SWI, and cold_content on a secondary y-axis using twinx()
fig, ax1 = plt.subplots(figsize=(12, 6))
em_basin_mean['snowmelt'].plot(ax=ax1, label='Snowmelt', marker='.', lw=1)
em_basin_mean['SWI'].plot(ax=ax1, label='SWI', marker='.', markersize=1, lw=1)
ax2 = ax1.twinx()
em_basin_mean['cold_content'].plot(ax=ax2, label='Cold Content', alpha=0.2, lw=2, color='orange') # indicative of snowpack melt phase
# em_basin_mean['sum_EB'].plot(ax=ax2, label='sumEB', color='orange') #backcheck on meltout
# Plot vertical lines at melt events (where sum_EB=0 and snowmelt>0)
melt_events = em_basin_mean.where((em_basin_mean['sum_EB'] == 0) & (em_basin_mean['snowmelt'] > 0), drop=True)
for melt_time in melt_events['time'].values:
    ax1.axvline(melt_time, color='red', linestyle='--', alpha=0.5)
ax1.set_xlabel('Time')
ax1.set_ylabel('Snowmelt (mm/day)')
# ax2.set_ylabel('Cold Content (kJ/m^2)')
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')

# Check on SWI vs. snowmelt

In [ ]:
# Ok, plot this same plot for one xy point
fig, ax1 = plt.subplots(figsize=(12, 6))
x=100
y=460
em_ds['snowmelt'].isel(x=x, y=y).plot(ax=ax1, label='Snowmelt', marker='.', lw=1, color='dodgerblue')
em_ds['SWI'].isel(x=x, y=y).plot(ax=ax1, label='SWI', lw=0.5, color='navy')
ax2 = ax1.twinx()
em_ds['cold_content'].isel(x=x, y=y).plot(ax=ax2, label='Cold Content', alpha=1, lw=2, color='orange') # indicative of snowpack melt phase
# em_ds['sum_EB'].isel(x=x, y=y).plot(ax=ax2, label='sumEB', color='orange') #backcheck on meltout

# Plot vertical lines at melt events (where sum_EB>0 and snowmelt>0 at this point
cond = ((em_ds['sum_EB'].isel(x=x, y=y) > 0) & (em_ds['snowmelt'].isel(x=x, y=y) > 0)).compute()
melt_times = em_ds['time'].values[cond.values]
for melt_time in melt_times:
    ax1.axvline(melt_time, color='salmon', linestyle='-', alpha=0.2)

# Mark melt initiation as the first instance SWI > 0, snowmelt > 0, and cold content is zero.
# The following week must also meet these conditions to avoid early season and transient melt events
init_cond = ((em_ds['SWI'].isel(x=x, y=y) > 0) & (em_ds['snowmelt'].isel(x=x, y=y) > 0) & (em_ds['cold_content'].isel(x=x, y=y) == 0)).compute()
# Implement the additional week condition using a leading rolling window
init_cond = init_cond.rolling(time=7, min_periods=7).min().shift(time=-6, fill_value=0).astype(bool)
init_time = em_ds['time'].values[init_cond.values][0]
ax1.axvline(init_time, color='r', linestyle='-', label='Melt Initiation')

# Mark meltout as the first instance after initiation where snowmelt = 0, sum_EB = 0, and cold_content = 0
meltout_cond = ((em_ds['snowmelt'].isel(x=x, y=y) == 0) & (em_ds['sum_EB'].isel(x=x, y=y) == 0) & (em_ds['cold_content'].isel(x=x, y=y) == 0)).compute()
# Restrict to times after melt initiation
meltout_cond = meltout_cond & (em_ds['time'] > init_time)
meltout_time = em_ds['time'].values[meltout_cond.values][0]
ax1.axvline(meltout_time, color='k', linestyle='-', label='Meltout')

ax1.set_ylabel('Snowmelt (mm/day)')
ax2.set_ylabel('Cold Content (kJ/m^2)')
# combine legends into one
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')

### Melt day heat map

In [ ]:
# Very cool. Ok, now that we have this, let's make a heat map of melt events over the entire domain (where sum_EB=0 and snowmelt>0)
# Count melt days per pixel over the water year
melt_cond = (em_ds['sum_EB'] > 0) & (em_ds['snowmelt'] > 0)
melt_days = melt_cond.sum(dim='time').compute()

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
melt_days.plot(ax=ax, cmap='YlOrRd',
            #    vmin=0, vmax=365,
               cbar_kwargs={'label': 'Melt days'})
ax.set_title(f'Melt days WY{wy}')
ax.set_aspect('equal')

In [ ]:
def plot_spatial_summary(da, wy,
                         map_title=None,
                         hist_title=None,
                         vmin=None, vmax=None,
                         cbar_label='',
                         bins=50,
                         hist_xlabel='',
                         sum_dim=None,
                         setfc='r',
                         figsize=(8, 4),
                         show_stats=True):
    """
    Spatial map + histogram side-by-side for any 2D summary DataArray.

    Parameters
    ----------
    da        : xr.DataArray — input array, 2D spatial or 3D if sum_dim provided
    wy        : int          — water year, used in default titles
    sum_dim   : str          — dimension to sum along before plotting (e.g. 'time')
    vmin/vmax : float        — colormap range for spatial map
    cbar_label: str          — colorbar label
    bins      : int or array — histogram bins passed to h.plot_hist
    hist_xlabel: str         — histogram x-axis label
    setfc     : str          — axes face color for masked/NaT pixels
    show_stats: bool         — overlay n, mean, median, std on histogram
    """
    if sum_dim is not None:
        da = da.sum(dim=sum_dim)

    # Compute once — reused for both plots and all stats
    da = da.compute()

    fig, axa = plt.subplots(1, 2, figsize=figsize)

    h.plot_one(da,
               title=map_title or f'WY{wy}',
               setfc=setfc,
               specify_ax=(fig, axa[0]),
               vmin=vmin, vmax=vmax,
               cbar_kwargs={'label': cbar_label},
               turnofflabels=True,
               turnoffaxes=True)

    h.plot_hist(da,
                bins=bins,
                specify_ax=(fig, axa[1]),
                xlabel=hist_xlabel,
                title=hist_title or f'WY{wy} distribution')

    if show_stats:
        stats_text = (
            f'n: {int((da > 0).sum().values)}\n'
            f'Mean: {float(da.mean().values):.1f}\n'
            f'Median: {float(da.median().values):.1f}\n'
            f'Std dev: {float(da.std().values):.1f}'
        )
        axa[1].text(0.95, 0.95, stats_text,
                    transform=axa[1].transAxes, fontsize=10,
                    verticalalignment='top', horizontalalignment='right',
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    plt.tight_layout()
    return fig, axa


In [ ]:
plot_spatial_summary(melt_days, wy,
                     map_title=f'Melt days WY{wy}', hist_title=f'Melt days distribution WY{wy}',
                     vmin=0, vmax=120, cbar_label='Melt days',
                     bins=np.arange(0, 121, 5), hist_xlabel='Melt days')

### Melt timing heat map (binary daily melt for domain, needs aggregation)

In [ ]:
# Derive function to determine melt timing for a data cube with dimensions (time, x, y)
def get_melt_init(ds):
    """
    Melt initiation: first date where SWI > 0, snowmelt > 0, and cold_content == 0
    for 7 consecutive days. This restriction is used to avoid early season and
    transient melt events.Pixels with no qualifying melt period are NaT.
    Default input ds derived from zarred em.nc daily output files

    Returns a (x, y) DataArray of melt initiation dates.
    """
    init_cond = ((ds['SWI'] > 0) & (ds['snowmelt'] > 0) & (ds['cold_content'] == 0)).compute()
    init_cond = init_cond.rolling(time=7, min_periods=7).min().shift(time=-6, fill_value=0).astype(bool)

    # argmax on a bool array returns the index of the first True per pixel
    first_idx = init_cond.argmax(dim='time')

    # argmax returns 0 for pixels where condition is never met — mask those out
    has_melt = init_cond.any(dim='time')

    return ds['time'].isel(time=first_idx).where(has_melt)

In [ ]:
melt_timing = get_melt_init(em_ds)
# melt_timing

In [ ]:
plot_spatial_summary(melt_timing.dt.dayofyear.where(melt_timing.notnull()), wy,
                     map_title=f'Melt initiation WY{wy}',
                     cbar_label='Melt initiation DOY',
                     bins=np.arange(1, 366, 5), hist_xlabel='Melt initiation DOY')

In [ ]:
# Ok, plot this same plot for one xy point
nomelt_pixel_ds = em_ds.isel(x=160,y=150)
fig, ax1 = plt.subplots(figsize=(12, 6))
nomelt_pixel_ds['snowmelt'].plot(ax=ax1, label='Snowmelt', marker='.', lw=1, color='dodgerblue')
nomelt_pixel_ds['SWI'].plot(ax=ax1, label='SWI', lw=0.5, color='navy')
ax2 = ax1.twinx()
nomelt_pixel_ds['cold_content'].plot(ax=ax2, label='Cold Content', alpha=1, lw=2, color='orange') # indicative of snowpack melt phase
# nomelt_pixel_ds['sum_EB'].plot(ax=ax2, label='sumEB', color='orange') #backcheck on meltout

ax1.set_ylabel('Snowmelt (mm/day)')
ax2.set_ylabel('Cold Content (kJ/m^2)')
# combine legends into one
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')

In [ ]:
# Ok, plot this same plot for one xy point
highmelt_pixel_ds = em_ds.isel(x=170,y=198)
fig, ax1 = plt.subplots(figsize=(12, 6))
highmelt_pixel_ds['snowmelt'].plot(ax=ax1, label='Snowmelt', marker='.', lw=1, color='dodgerblue')
highmelt_pixel_ds['SWI'].plot(ax=ax1, label='SWI', lw=0.5, color='navy')
ax2 = ax1.twinx()
highmelt_pixel_ds['cold_content'].plot(ax=ax2, label='Cold Content', alpha=1, lw=2, color='orange') # indicative of snowpack melt phase
# highmelt_pixel_ds['sum_EB'].plot(ax=ax2, label='sumEB', color='orange') #backcheck on meltout

ax1.set_ylabel('Snowmelt (mm/day)')
ax2.set_ylabel('Cold Content (kJ/m^2)')
# combine legends into one
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')

### Meltout timing heat map

In [ ]:
# Derive function to determine melt timing for a data cube with dimensions (time, x, y)
def get_meltout(ds, melt_timing):
    """
    Meltout timing: first date after melt initiation where snowmelt=0, sum_EB=0, and cold_content=0
    Pixels with no qualifying melt period are NaT.
    Default input ds derived from zarred em.nc daily output files

    Returns a (x, y) DataArray of meltout dates.
    """
    meltout_cond = (
        (ds['sum_EB'] == 0) & (ds['snowmelt'] == 0) & (ds['cold_content'] == 0)
    ).compute()

    # Restrict to times after melt initiation
    meltout_cond = meltout_cond & (meltout_cond.time > melt_timing)

    # argmax on a bool array returns the index of the first True per pixel
    first_idx = meltout_cond.argmax(dim='time')

    # argmax returns 0 for pixels where condition is never met — mask those out
    has_meltout = meltout_cond.any(dim='time')

    return ds['time'].isel(time=first_idx).where(has_meltout)

In [ ]:
meltout_timing = get_meltout(em_ds, melt_timing)
# meltout_timing

In [ ]:
plot_spatial_summary(meltout_timing.dt.dayofyear.where(meltout_timing.notnull()), wy,
                     map_title=f'Meltout WY{wy}',
                     cbar_label='Meltout DOY',
                     bins=np.arange(1, 366, 5), hist_xlabel='Meltout DOY')

In [ ]:
# Combine the melt initiation and meltout timing to derive a melt season 
# This is an oversimplistic definition of melt season, but captures the bounds of 
# melt onset to meltout, which is when water is being released
# Future plots will focus on daily melt contributions

# The melt season dataset will be in the form of (time, x, y)
# It will be generated as a boolean value of True between melt initiation and meltout, and False elsewhere
# Combine melt initiation and meltout into a single dataset
during_melt_season = (
    (em_ds['time'] >= melt_timing) & (em_ds['time'] <= meltout_timing)
)

melt_season_ds = during_melt_season.rename('during_melt_season').to_dataset()

# Should see True values only between melt_timing and meltout_timing at this point
x, y = 100, 100
print(melt_timing.isel(x=x, y=y).values)
print(meltout_timing.isel(x=x, y=y).values)
print(melt_season_ds['during_melt_season'].isel(x=x, y=y).sum().values, 'melt days')

In [ ]:
plot_spatial_summary(melt_season_ds['during_melt_season'], wy,
                     sum_dim='time',
                     map_title=f'Melt season duration WY{wy}',
                     vmin=0, vmax=100, cbar_label='days',
                     bins=np.arange(1, 366, 5), hist_xlabel='Melt season duration (days)')

In [ ]:
# Pull in terrain now, you will need this for all of the other plots
terrain_fn = h.fn_list(script_dir, f'{basin}_setup/output_100m/topo.nc')[0]
# Load in only the elevation variable to save memory
terrain_ds = xr.open_dataset(terrain_fn, drop_variables=[var for var in xr.open_dataset(terrain_fn).data_vars if var != 'dem'])
h.plot_one(terrain_ds['dem'], title=f'{basin.capitalize()} elevation', cmap='plasma', figsize=(3, 5))

In [ ]:
plot_spatial_summary(terrain_ds['dem'], wy,
                     sum_dim=None,
                     figsize=(10, 4),
                     map_title=f'{basin.capitalize()} elevation',
                     cbar_label='masl',
                     hist_xlabel='Elevation (masl)')
plt.tight_layout()

In [ ]:
# Plot a heatmap of melt season duration based on elevation
# First generate the elevation bins with standard increments
bin_step = 200 # this should be good
# Construct the bins to cover the full elevation range of the basin, rounded to the nearest 100 m
elev_bins = np.arange(np.round(terrain_ds['dem'].min().values, -2), np.round(terrain_ds['dem'].max().values, -2) + bin_step, bin_step)

# Flatten both to 1D — pixels are the unit of analysis
elev_flat  = terrain_ds['dem'].values.ravel()
duration_flat = melt_season_ds['during_melt_season'].sum(dim='time').values.ravel()

# Keep only pixels with valid elevation and non-zero melt season
valid = (duration_flat > 0) & np.isfinite(elev_flat)

fig, ax = plt.subplots(figsize=(6, 3))
h2d, xedges, yedges, img = ax.hist2d(
    duration_flat[valid],
    elev_flat[valid],
    bins=[np.arange(0, 100, 5), elev_bins],
    cmap='YlOrRd'
)
plt.colorbar(img, ax=ax, label='Frequency')
ax.set_xlabel(f'Melt season duration (WY {wy})')
ax.set_ylabel(f'{basin.capitalize()} Elevation (m)')

In [ ]:
# Plot a similar heatmap for duration, but use time on the x-axis
# Flatten spatial dims → (time, pixels)
during_flat = melt_season_ds['during_melt_season'].values.reshape(len(melt_season_ds['time']), -1)
elev_flat   = terrain_ds['dem'].values.ravel()

# Assign each pixel to an elevation bin
elev_bin_idx = np.digitize(elev_flat, elev_bins) - 1
n_bins = len(elev_bins) - 1
valid = np.isfinite(elev_flat) & (elev_bin_idx >= 0) & (elev_bin_idx < n_bins)

# Count melting pixels per (time, elev_bin) using matrix multiply
one_hot = np.zeros((valid.sum(), n_bins))
one_hot[np.arange(valid.sum()), elev_bin_idx[valid]] = 1

# number of melting pixels per time step and elevation bin
melt_count_matrix = during_flat[:, valid].astype(float) @ one_hot  # (365, n_bins)
melt_frac_matrix = melt_count_matrix.copy()

# Normalize to fraction of pixels melting per bin (removes elevation band size bias)
melt_frac_matrix /= one_hot.sum(axis=0)

In [ ]:
sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/ucrb-isnobal/')
from scripts.thp.melt_plots import time_to_dowy, get_dowy_month_ticks

In [ ]:
dowy = time_to_dowy(melt_season_ds['time'].values, wy)
xticks, xtick_labels = get_dowy_month_ticks(wy)

# # compute doy
# doy = melt_season_ds['time'].dt.dayofyear.values

# # change x-ticks to the beginning of the Month and Day rather than day of year
# xticks = [1, 32, 60, 91, 121, 152, 182, 213, 244, 274, 305, 335]
# xtick_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

In [ ]:
# Plot
fig, ax = plt.subplots(figsize=(6, 3))
c = ax.pcolormesh(dowy, elev_bins[:-1], melt_frac_matrix.T, cmap='YlOrRd', vmin=0, vmax=1)
plt.colorbar(c, ax=ax, label='Fraction of pixels melting')

ax.set_xticks(xticks)
ax.set_xticklabels(xtick_labels)
# Trim
ax.set_xlim(0, 365)
ax.set_xlabel(f'WY {wy}')
ax.set_ylabel(f'{basin.capitalize()} Elevation (m)')

In [ ]:
# Plot
fig, ax = plt.subplots(figsize=(6, 3))
c = ax.pcolormesh(dowy, elev_bins[:-1], melt_frac_matrix.T, cmap='cividis', vmin=0, vmax=1)
plt.colorbar(c, ax=ax, label='Fraction of pixels melting')
# change x-ticks to the beginning of the Month and Day rather than day of year
ax.set_xticks(xticks)
ax.set_xticklabels(xtick_labels)
# Trim
ax.set_xlim(0, 365)
ax.set_xlabel(f'WY {wy}')
ax.set_ylabel(f'{basin.capitalize()} Elevation (m)')

In [ ]:
# Plot the same, but representing number of melting pixels, without normalization

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
c = ax.pcolormesh(dowy, elev_bins[:-1], melt_count_matrix.T, cmap='YlOrRd')
plt.colorbar(c, ax=ax, label='Melting pixels')
# change x-ticks to the beginning of the Month and Day rather than day of year
ax.set_xticks(xticks)
ax.set_xticklabels(xtick_labels)
# Trim
ax.set_xlim(0, 365)
ax.set_xlabel(f'WY {wy}')
ax.set_ylabel(f'{basin.capitalize()} Elevation (m)')
# # Save this to file
# outdir = '/uufs/chpc.utah.edu/common/home/u6058223/public_html/thp_update/figures'
# outname = f'{outdir}/melt_plots/{basin}_melt_season_heatmap_freq_wy{wy}.png'
# # make parent directories, if needed
# import os
# os.makedirs(os.path.dirname(outname), exist_ok=True)
# plt.savefig(f'{outname}', dpi=300, bbox_inches='tight')

In [ ]:
counts, _ = np.histogram(elev_flat[np.isfinite(elev_flat)], bins=elev_bins)
area_frac = counts / counts.sum()  # fraction of total basin area per bin

fig, ax = plt.subplots(figsize=(3, 6))
ax.barh(elev_bins[:-1], area_frac,
        height=bin_step * 0.9, align='edge', alpha=0.3, color='steelblue', edgecolor='w')
ax.set_ylabel('Elevation (m)')
ax.set_xlabel('Fraction of basin area')
ax.set_title(f'Hypsometry — {basin}')
ax.set_xlim(0, area_frac.max() * 1.1)

# Optional: cumulative on twin axis
ax2 = ax.twiny()
ax2.plot(np.cumsum(area_frac), elev_bins[:-1] + bin_step / 2,
         color='dodgerblue', lw=2.5)
ax2.set_xlim(0, 1)
ax2.set_xlabel('Cumulative fraction')


In [ ]:
fig, (ax_hyps, ax) = plt.subplots(1, 2, figsize=(8, 4),
                                   sharey=True,
                                   gridspec_kw={'width_ratios': [1, 4]})

# Hypsometric curve (left)
counts, _ = np.histogram(elev_flat[np.isfinite(elev_flat)], bins=elev_bins)
ax_hyps.barh(elev_bins[:-1], counts / counts.sum(),
             height=bin_step * 0.9, align='edge', alpha=0.3, color='steelblue', edgecolor='w')
ax_hyps.set_xlabel('Basin\narea fraction')
ax_hyps.set_ylabel(f'{basin.capitalize()} Elevation (m)')

ax_hyps2 = ax_hyps.twiny()
ax_hyps2.plot(np.cumsum(counts / counts.sum()), elev_bins[:-1] + bin_step / 2,
              color='dodgerblue', lw=2.5)
ax_hyps2.set_xlim(0, 1)
ax_hyps2.set_xlabel('Cumulative fraction')

# Heatmap (right)
c = ax.pcolormesh(dowy, elev_bins[:-1], melt_count_matrix.T, cmap='cividis')
plt.colorbar(c, ax=ax, label='Melting pixels', pad=0.01)
ax.set_xticks(xticks)
ax.set_xticklabels(xtick_labels)
ax.set_xlim(0, 365)
ax.set_xlabel(f'WY {wy}')
ax.tick_params(labelleft=False)  # y-labels now on hyps panel

ax.set_ylim(elev_bins[0], elev_bins[-1] - bin_step / 2)

plt.tight_layout()


### SWI contributions plot

In [ ]:
import numpy as np
bin_step = 200
np.arange(1300, 4500 + bin_step, bin_step)

In [ ]:
# Following the plot of melting pixels over time and by elevation,
# Compute the SWI contribution per elevation band for melting pixels only
# this is not just during the melting season, but for only the melting days
# So use a conditional that first isolates melt days alone
# Then apply the same matrix multiplication approach to sum SWI per elevation bin per day
# then plot as a heatmap like above
# Can use the melt condition from above where sum_EB > 0 and snowmelt > 0

# Zero out non-melt days to remove from further operations
# Dims: (time=365, y, x)
swi_during_melt_ds = em_ds['SWI'].where(melt_cond, 0)

# Collapse spatial dimensions into 1d for mat mult below
# Dims: (time=365, y*x)
swi_flat = swi_during_melt_ds.values.reshape(len(swi_during_melt_ds['time']), -1)

# Count SWI contribution per (time, elev_bin) using matrix multiply
# recall that one_hot is a binary matrix where denoting whether the
# pixel belongs to a given elevation bin
# Dims: (time=365, y*x pixels) @ (y*x pixels, n_elevbins) → (time=365, n_elevbins)
swi_matrix = swi_flat[:, valid] @ one_hot

In [ ]:
# Plot total SWI in meters
fig, ax = plt.subplots(figsize=(6, 3))
c = ax.pcolormesh(dowy, elev_bins[:-1], swi_matrix.T / 1000,
                     cmap='YlGnBu')
plt.colorbar(c, ax=ax, label='Total SWI contribution (m)', pad=0.02)
# change x-ticks to the beginning of the Month and Day rather than day of year
ax.set_xticks(xticks)
ax.set_xticklabels(xtick_labels)
# Trim
ax.set_xlim(0, 365)
ax.set_xlabel(f'WY {wy}')
ax.set_ylabel(f'{basin.capitalize()} Elevation (m)')

# TODO: turn this into a function that can be applied to other variables and basins, with options for normalization, etc. then move to melt_plots.py

- per pixel: mm (that day, starting raster)
- per elevation band: total mm (height) that day (all melting pixels * per pixel mm)
    - meter conversion ^ / 1000 (1 m / 1000 mm) total meters that day for a given pixel size 
- volume per elevation band: total height in meters (all melting pixels * per pixel melt) * pixel area in meters(for all melting pixels) --> cubic meters
    - TAF conversion ^ * 0.000810714 acreft / 1 cubic meter * 1 TAF / 1000 acreft
- normalized per elevation band: average melt per melting pixel in that band (total mm / # melting pixels)

In [ ]:
# Convert SWI to thousands of acre-feet (TAF) per elevation bin
# 1 m^3 = 0.000810713 acre-foot
# 1 pixel area = 100 m * 100 m = 10,000 m^2
# So multiply SWI in m by pixel area in m^2 to get volume in m^3, then convert to thousands of acre-feet
pixel_area_m2 = 100 * 100
swi_taf_matrix = swi_matrix * pixel_area_m2 * 0.000810713 / 1000
# Plot total SWI in TAF
fig, ax = plt.subplots(figsize=(6, 3))
c = ax.pcolormesh(dowy, elev_bins[:-1], swi_taf_matrix.T,
                     cmap='YlGnBu')
plt.colorbar(c, ax=ax, label='Total SWI contribution (TAF)', pad=0.02)
# change x-ticks to the beginning of the Month and Day rather than day of year
ax.set_xticks(xticks)
ax.set_xticklabels(xtick_labels)
# Trim
ax.set_xlim(0, 365)
ax.set_xlabel(f'WY {wy}')
ax.set_ylabel(f'{basin.capitalize()} Elevation (m)')

In [ ]:
# Plot total SWI in meters with basin mean SWI overlaid
fig, ax = plt.subplots(figsize=(6, 5))
c = ax.pcolormesh(dowy, elev_bins[:-1], swi_matrix.T / 1000,
                     cmap='YlGnBu')
# Put the colorbar horizontally to make room for the secondary y-axis
plt.colorbar(c, ax=ax, label='Total SWI contribution (m)', pad=0.15,
             orientation='horizontal', anchor=(0.5, 1.2))
# change x-ticks to the beginning of the Month and Day rather than day of year
ax.set_xticks(xticks)
ax.set_xticklabels(xtick_labels)
# Trim
ax.set_xlim(0, 365)
ax.set_xlabel(f'WY {wy}')
ax.set_ylabel(f'{basin.capitalize()} Elevation (m)')
# Plot basin_mean SWI on top using dowy for x-axis and a secondary y-axis
ax2 = ax.twinx()
swi_basin_mean = em_basin_mean['SWI'].values / 1000  # convert to meters
ax2.plot(dowy, swi_basin_mean, color='b', lw=0.35, label='Basin-mean SWI')
# color the y-axis ticks to match the line
ax2.tick_params(axis='y', colors='b')
ax2.set_ylabel('Basin-mean SWI (m)')

In [ ]:
# Now normalize it by the number of melting pixels per bin
# to get the average SWI contribution on melt days per melting pixel
# Avoid divide by zero by setting zeros to NaN temporarily
swi_matrix_norm = swi_matrix / np.where(melt_count_matrix == 0, np.nan, melt_count_matrix)

# Re-plot, but limit the cbar stretch between the 5-95th percentile
fig, ax = plt.subplots(figsize=(6, 3))
c = ax.pcolormesh(dowy, elev_bins[:-1], swi_matrix_norm.T, cmap='YlGnBu', vmin=0, vmax=1e2)
plt.colorbar(c, ax=ax, label='Average melt day \nSWI contribution (mm)')
# change x-ticks to the beginning of the Month and Day rather than day of year
ax.set_xticks(xticks)
ax.set_xticklabels(xtick_labels)
plt.tight_layout()

In [ ]:
# Plot a histogram of the average SWI contribution on melt days per pixel, separated by elevation band
# First flatten the normalized matrix to 1D and create a corresponding elevation bin array
swi_flat_norm = swi_matrix_norm.T.ravel()
elev_bin_flat = np.repeat(elev_bins[:-1], len(dowy))
# Create a DataFrame for seaborn
df = pd.DataFrame({
    'Average SWI on melt days (mm)': swi_flat_norm,
    'Elevation bin': pd.Categorical(elev_bin_flat, categories=elev_bins[:-1], ordered=True)
    })
df

In [ ]:
# Plot
plt.figure(figsize=(8, 4))
sns.boxplot(x='Elevation bin', y='Average SWI on melt days (mm)',
            data=df, color='dodgerblue', width=0.25)
# Round xtick labels to the nearest 100 m
plt.xticks(ticks=np.arange(len(elev_bins[:-1])), labels=[f'{int(elev)}' for elev in elev_bins[:-1]],
           rotation=45
           )
plt.ylim(-1e1, 1.5e2)
plt.xlabel('Elevation bin (m)')
plt.title(f'Melt day average SWI contributions in the {basin.capitalize()} (WY {wy})')
plt.ylabel('Per-pixel average SWI contribution (mm)')
plt.tight_layout()
# TODO split this up by month (1 month per subplot of elevation boxplots)
# to see how the relationship evolves
# TODO boxplot the entire season, month by month, split by elevation band

### EB melt contributions plot
Plot the top daily contributor by elevation

In [ ]:
from scripts.thp.melt_plots import plot_eb_term_heatmap, plot_dominant_eb_term, LOGGER
import os

In [ ]:
# read in median term matrices
ow = False
zarr_dir = os.path.join(workdir, 'zarr_stores')
eb_terms_active = ['net_solar', 'net_LW', 'sensible_heat',
                           'latent_heat', 'snow_soil', 'precip_advected']
prop_suffix = 'split'
basin = 'jordan'
wy = 2024
prop_fn = f'{zarr_dir}/{basin}_eb_proportions_{prop_suffix}_wy{wy}.zarr'
median_fn = prop_fn.replace('.zarr', '_medians.nc')

if os.path.exists(median_fn) and not ow:
    LOGGER.info('Loading cached EB median matrices: %s', median_fn)
    median_ds = xr.open_dataset(median_fn)
    term_median_matrices = {
        t: median_ds['median_fraction'].sel(term=t).values
        for t in eb_terms_active
    }

### EB melt heat maps (intensity not by frequency, but proportional value)

In [ ]:
# Create default elev bins
bin_step = 200
elev_bins = np.arange(0, 4400 + bin_step, bin_step)
elev_bins

In [ ]:
plot_eb_term_heatmap(basin=basin, wy=wy, elev_bins=elev_bins, median_matrix=term_median_matrices['net_solar'], term_name='net_solar', dowy=dowy)

### EB dominant term

In [ ]:
plot_dominant_eb_term(basin=basin, wy=wy, elev_bins=elev_bins,
                      term_median_matrices=term_median_matrices, dowy=dowy)